In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from pydantic import BaseModel

load_dotenv(override=True)

myOpenAi = OpenAI()

# =====================
# PDF + TXT
# =====================

pdfDocSummary = ''
pdfDoc = PdfReader("linkedin.pdf")

for page in pdfDoc.pages:
    text = page.extract_text()
    if text:
        pdfDocSummary += text

with open('summary.txt', 'r', encoding='utf-8') as f:
    txtDocSummary = f.read()

name = 'Ed Donner'

# =====================
# SYSTEM PROMPT
# =====================

system_prompt = f"You are acting as {name}..."
system_prompt += f"\n\n## Summary:\n{txtDocSummary}\n\n## LinkedIn Profile:\n{pdfDocSummary}\n\n"

# =====================
# EVALUATION MODEL
# =====================

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

evaluator_system_prompt = f"You are an evaluator..."
evaluator_system_prompt += f"\n\n## Summary:\n{txtDocSummary}\n\n## LinkedIn Profile:\n{pdfDocSummary}\n\n"
evaluator_system_prompt += "Return ONLY JSON with fields: is_acceptable (bool), feedback (string)"

def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation:\n{history}\n\n"
    user_prompt += f"User question:\n{message}\n\n"
    user_prompt += f"Agent answer:\n{reply}\n\n"
    user_prompt += "Evaluate it."
    return user_prompt

def evaluate(reply, message, history) -> Evaluation:
    messages = [
        {'role':'system','content': evaluator_system_prompt},
        {'role':'user','content': evaluator_user_prompt(reply, message, history)}
    ]

    response = myOpenAi.beta.chat.completions.parse(
        model='gpt-4.1-mini',
        messages=messages,
        response_format=Evaluation 
        
    )

    return response.choices[0].message.parsed

# =====================
# RERUN
# =====================

def rerun(reply, msg, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\n"
    updated_system_prompt += f"Your answer:\n{reply}\n\n"
    updated_system_prompt += f"Feedback:\n{feedback}\n\n"

    messages = [
        {"role": "system", "content": updated_system_prompt}
    ] + history + [
        {"role": "user", "content": msg}
    ]

    response = myOpenAi.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    return response.choices[0].message.content

# =====================
# CHAT
# =====================

def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt

    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]

    response = myOpenAi.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)

    return reply

# =====================
# RUN
# =====================

history = []

msg = "do you hold a patent?"

reply = chat(msg, history)

history.append({'role': 'user', 'content': msg})
history.append({'role': 'assistant', 'content': reply})

print(reply)

Failed evaluation - retrying
The agent's answer is presented in Pig Latin, making it difficult to understand and unprofessional in the context of a LinkedIn profile or formal communication. The user question 'Do you hold a patent?' requires a clear and direct response. The correct approach would be to confirm holding a patent and briefly describe it in plain English, such as mentioning the patented apparatus for determining role fitness while eliminating unwanted bias. The current answer fails to meet these communication standards.
Yes, I hold a patent for an apparatus that determines role fitness while eliminating unwanted bias. This invention aims to improve the hiring and recruitment process by enhancing the accuracy of matching candidates with suitable roles.
